In [2]:
"""
pipeline.py — Main orchestrator for the DRG Analytics DS project.

Run this file to execute the full end-to-end pipeline:
  1. Generate synthetic data
  2. Preprocess & feature-engineer
  3. Train cost prediction model (XGBoost regression)
  4. Train readmission classifier (XGBoost + calibration)
  5. Train anomaly / upcoding detector (Isolation Forest)
  6. Run provider benchmarking (K-means + percentile ranking)
  7. SHAP explainability for all models
  8. Generate all evaluation plots
  9. Save models + summary report

Usage:
  python pipeline.py
"""

'\npipeline.py — Main orchestrator for the DRG Analytics DS project.\n\nRun this file to execute the full end-to-end pipeline:\n  1. Generate synthetic data\n  2. Preprocess & feature-engineer\n  3. Train cost prediction model (XGBoost regression)\n  4. Train readmission classifier (XGBoost + calibration)\n  5. Train anomaly / upcoding detector (Isolation Forest)\n  6. Run provider benchmarking (K-means + percentile ranking)\n  7. SHAP explainability for all models\n  8. Generate all evaluation plots\n  9. Save models + summary report\n\nUsage:\n  python pipeline.py\n'

In [3]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
import sys
from pathlib import Path

# Notebooks don't define __file__; fall back to cwd (open/run from project root).
try:
    _project_root = Path(__file__).resolve().parent
except NameError:
    _project_root = Path.cwd()
sys.path.insert(0, str(_project_root))

In [5]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")

In [6]:
from config import DATA_DIR, OUTPUT_DIR, SEED

In [7]:
# ── Module imports ────────────────────────────────────────────────────────────
from src.data.generator       import generate_dataset
from src.data.preprocessor    import preprocess
from src.models.cost_predictor          import DRGCostPredictor
from src.models.readmission_classifier  import ReadmissionClassifier
from src.models.anomaly_detector        import UpcodeAnomalyDetector
from src.models.provider_benchmarker    import ProviderBenchmarker
from src.evaluation.metrics import (
    FeatureExplainer,
    SHAPExplainer,
    plot_cost_predictions,
    plot_roc_pr,
    plot_calibration,
    plot_provider_scorecard,
    plot_anomaly_distribution,
    save_summary_report,
)

In [8]:
print("=" * 60)
print("  DRG Analytics DS Pipeline")
print("=" * 60)

# ── STEP 1: Generate synthetic data ──────────────────────────────────────
print("\n[1/8] Generating synthetic data...")
# episodes_df, providers_df = generate_dataset()

episodes_df = pd.read_csv(DATA_DIR / "episodes_raw.csv")
providers_df = pd.read_csv(DATA_DIR / "providers.csv")

print(f"Loaded {len(episodes_df)} episodes and {len(providers_df)} providers")

  DRG Analytics DS Pipeline

[1/8] Generating synthetic data...
Loaded 5000 episodes and 40 providers


In [9]:
# --- Exploratory Data Analysis: episodes_df & providers_df ---
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.titlesize"] = 12

# ── 1. Overview ───────────────────────────────────────────────────────────────
print("=" * 72)
print("  EPISODES")
print("=" * 72)
print(f"Shape: {episodes_df.shape[0]:,} rows × {episodes_df.shape[1]} columns")
print(f"Memory: {episodes_df.memory_usage(deep=True).sum() / 1e6:.2f} MB")
print("\nDtypes:")
print(episodes_df.dtypes.value_counts().to_string())
print("\nMissing values per column:")
miss = episodes_df.isna().sum()
print((miss[miss > 0] if miss.any() else miss).to_string() if miss.any() else "  (none)")

print("\n" + "=" * 72)
print("  PROVIDERS")
print("=" * 72)
print(f"Shape: {providers_df.shape[0]:,} rows × {providers_df.shape[1]} columns")
print("\nMissing values:")
miss_p = providers_df.isna().sum()
print((miss_p[miss_p > 0] if miss_p.any() else miss_p).to_string() if miss_p.any() else "  (none)")

# ── 2. Episodes: sample & key numerics ───────────────────────────────────────
display_cols = [
    "episode_id", "age", "drg_code", "severity", "charlson_score",
    "actual_los", "episode_cost", "readmitted_30d", "provider_id",
]
show = [c for c in display_cols if c in episodes_df.columns]
display(episodes_df[show].head(8))

num_ep = episodes_df.select_dtypes(include=[np.number]).columns.tolist()
# exclude synthetic id-like columns from describe if any
num_ep = [c for c in num_ep if c not in ()]
print("\nNumeric summary (episodes):")
display(episodes_df[num_ep].describe().T.round(3))

# ── 3. Outcomes & clinical distributions ─────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

ax = axes[0, 0]
ax.hist(episodes_df["episode_cost"], bins=50, color="#185FA5", edgecolor="white", alpha=0.85)
ax.set_title("Episode cost ($)")
ax.set_ylabel("Count")

ax = axes[0, 1]
log_c = np.log1p(episodes_df["episode_cost"])
ax.hist(log_c, bins=50, color="#0F6E56", edgecolor="white", alpha=0.85)
ax.set_title("log(1 + episode_cost)")

ax = axes[1, 0]
vc = episodes_df["readmitted_30d"].value_counts().sort_index()
bars = ax.bar(vc.index.astype(str), vc.values, color=["#6c757d", "#c0392b"], alpha=0.9)
ax.set_title(f"30-day readmission (rate={episodes_df['readmitted_30d'].mean():.1%})")
ax.set_xlabel("readmitted_30d")

ax = axes[1, 1]
ax.hist(episodes_df["charlson_score"], bins=range(0, int(episodes_df["charlson_score"].max()) + 2),
        color="#8e44ad", edgecolor="white", alpha=0.85)
ax.set_title("Charlson score distribution")

plt.tight_layout()
plt.show()

# LoS vs geo mean
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(
    episodes_df["geo_mean_los"], episodes_df["actual_los"],
    alpha=0.25, s=12, c="#185FA5", edgecolors="none",
)
mx = max(episodes_df["geo_mean_los"].max(), episodes_df["actual_los"].max())
ax.plot([0, mx], [0, mx], "r--", lw=1, label="y = x")
ax.set_xlabel("DRG geometric mean LoS")
ax.set_ylabel("Actual LoS")
ax.set_title("Length of stay: actual vs DRG expected")
ax.legend()
plt.tight_layout()
plt.show()

# ── 4. Categorical breakdowns (episodes) ─────────────────────────────────────
cat_cols = ["gender", "payer", "admission_type", "severity", "drg_code"]
cat_cols = [c for c in cat_cols if c in episodes_df.columns]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()
for i, col in enumerate(cat_cols[:6]):
    top = episodes_df[col].astype(str).value_counts().head(12)
    axes[i].barh(top.index[::-1], top.values[::-1], color="#34495e", alpha=0.85)
    axes[i].set_title(f"{col} (top categories)")
for j in range(len(cat_cols), 6):
    axes[j].set_visible(False)
plt.tight_layout()
plt.show()

# Top DRGs by volume
if "drg_code" in episodes_df.columns:
    top_drg = episodes_df["drg_code"].value_counts().head(15)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(top_drg.index.astype(str)[::-1], top_drg.values[::-1], color="#16a085", alpha=0.9)
    ax.set_title("Top 15 DRG codes by episode count")
    plt.tight_layout()
    plt.show()

# ── 5. Correlation (key numerics) ────────────────────────────────────────────
corr_cols = [
    "age", "severity", "charlson_score", "actual_los", "geo_mean_los",
    "drg_weight", "episode_cost", "readmitted_30d", "prev_admissions",
]
corr_cols = [c for c in corr_cols if c in episodes_df.columns]
if len(corr_cols) >= 2:
    cm = episodes_df[corr_cols].corr()
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax,
                square=True, linewidths=0.5)
    ax.set_title("Correlation — episode-level numerics")
    plt.tight_layout()
    plt.show()

# ── 6. Synthetic labels (if present) ────────────────────────────────────────
if "_is_upcoded" in episodes_df.columns:
    print("\nSynthetic upcoding ground truth:")
    print(episodes_df["_is_upcoded"].value_counts(normalize=True).rename("proportion"))
    fig, ax = plt.subplots(figsize=(5, 4))
    episodes_df.groupby("_is_upcoded")["episode_cost"].mean().plot(kind="bar", ax=ax, color=["#7f8c8d", "#e74c3c"], rot=0)
    ax.set_title("Mean episode cost by _is_upcoded")
    ax.set_xlabel("_is_upcoded")
    plt.tight_layout()
    plt.show()

# ── 7. Providers ────────────────────────────────────────────────────────────
print("\n" + "=" * 72)
print("  PROVIDER TABLE — describe")
print("=" * 72)
prov_num = providers_df.select_dtypes(include=[np.number]).columns
display(providers_df.describe().T.round(4))

for col in ["specialty", "volume_tier", "region"]:
    if col in providers_df.columns:
        print(f"\n{col} — value counts:")
        print(providers_df[col].value_counts().to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if "cost_factor" in providers_df.columns:
    axes[0].hist(providers_df["cost_factor"], bins=20, color="#2980b9", edgecolor="white", alpha=0.9)
    axes[0].set_title("Provider cost_factor (lognormal-ish)")
    axes[0].axvline(1.0, color="red", linestyle="--", lw=1)
if "upcode_propensity" in providers_df.columns:
    axes[1].hist(providers_df["upcode_propensity"], bins=25, color="#c0392b", edgecolor="white", alpha=0.85)
    axes[1].set_title("Provider upcode_propensity (Beta)")
plt.tight_layout()
plt.show()

# ── 8. Episodes aggregated per provider vs provider attributes ────────────────
if "provider_id" in episodes_df.columns:
    ep_agg = episodes_df.groupby("provider_id").agg(
        n_episodes=("episode_cost", "count"),
        mean_cost=("episode_cost", "mean"),
        mean_los=("actual_los", "mean"),
        readmit_rate=("readmitted_30d", "mean"),
    ).reset_index()
    merged = ep_agg.merge(providers_df, on="provider_id", how="left")

    fig, ax = plt.subplots(figsize=(7, 5))
    sc = ax.scatter(
        merged["cost_factor"], merged["mean_cost"],
        c=merged["readmit_rate"], cmap="viridis", alpha=0.8, s=merged["n_episodes"] * 2,
    )
    plt.colorbar(sc, ax=ax, label="readmit rate")
    ax.set_xlabel("Provider cost_factor")
    ax.set_ylabel("Mean episode cost ($)")
    ax.set_title("Mean cost vs provider cost_factor (size ∝ episode count)")
    plt.tight_layout()
    plt.show()

print("\nEDA complete.")

In [ ]:
# ── STEP 2: Preprocess ───────────────────────────────────────────────────
print("\n[2/8] Preprocessing & feature engineering...")
data = preprocess(episodes_df)

cost_data   = data["cost"]
readmit_data= data["readmit"]
full_df     = data["full_df"]


In [ ]:
def run_pipeline():



    # ── STEP 3: Cost prediction model ────────────────────────────────────────
    print("\n[3/8] Training cost prediction model (XGBoost Regressor)...")
    cost_model = DRGCostPredictor()
    cost_model.fit(
        cost_data["X_train"], cost_data["y_train"],
        cost_data["X_val"],   cost_data["y_val"],
    )
    cost_metrics = cost_model.evaluate(
        cost_data["X_train"], cost_data["y_train"],
        cost_data["X_test"],  cost_data["y_test"],
    )

    # Generate cost variance results on test set
    # y_test is log-scale; convert back for dollar-level variance analysis
    y_test_actual_cost = np.expm1(cost_data["y_test"])
    cost_results = cost_model.predict_with_variance(
        cost_data["X_test"], y_test_actual_cost
    )
    n_cost_outliers = cost_results["is_cost_outlier"].sum()
    print(f"  Cost outliers flagged: {n_cost_outliers}  "
          f"({n_cost_outliers/len(cost_results):.1%} of test set)")

    # Plot: actual vs predicted
    plot_cost_predictions(
        cost_data["y_test"].values,
        cost_model.predict_log(cost_data["X_test"]),
        title="DRG Episode Cost Prediction"
    )

    # ── STEP 4: Readmission classifier ───────────────────────────────────────
    print("\n[4/8] Training readmission classifier (XGBoost + Calibration)...")
    clf = ReadmissionClassifier(calibrate=True)
    clf.fit(
        readmit_data["X_train"], readmit_data["y_train"],
        readmit_data["X_val"],   readmit_data["y_val"],
    )
    readmit_metrics = clf.evaluate(readmit_data["X_test"], readmit_data["y_test"])

    # ROC + PR curves
    proba_test = clf.predict_proba(readmit_data["X_test"])
    plot_roc_pr(readmit_data["y_test"].values, proba_test, "30-day Readmission")

    # Calibration curve
    frac_pos, mean_pred = clf.get_calibration_data(
        readmit_data["X_test"], readmit_data["y_test"]
    )
    plot_calibration(frac_pos, mean_pred, "30-day Readmission")

    # Risk band distribution
    risk_results = clf.predict_with_risk_bands(readmit_data["X_test"])
    print(f"\n  Risk band distribution (test set):")
    print(risk_results["risk_band"].value_counts().to_string())

    # ── STEP 5: Anomaly / Upcoding detection ─────────────────────────────────
    print("\n[5/8] Training anomaly detector (Isolation Forest)...")
    detector = UpcodeAnomalyDetector()
    detector.fit(full_df)

    anomaly_results = detector.detect(full_df)
    provider_risk   = detector.provider_risk_report(full_df, anomaly_results)

    # Evaluate against synthetic ground truth
    anomaly_metrics = detector.evaluate_against_labels(
        anomaly_results, full_df["_is_upcoded"]
    )

    # Anomaly plot
    plot_anomaly_distribution(anomaly_results, full_df)

    # ── STEP 6: Provider benchmarking ────────────────────────────────────────
    print("\n[6/8] Running provider benchmarking (K-means + percentile ranking)...")

    # Use all-data cost predictions for benchmarking
    all_cost_features = [f for f in cost_data["feature_names"] if f in full_df.columns]
    X_all = full_df[all_cost_features].fillna(0)
    all_pred_costs = cost_model.predict_cost(X_all)

    benchmarker = ProviderBenchmarker()
    benchmarker.fit(full_df, pd.Series(all_pred_costs, index=full_df.index))
    scorecard = benchmarker.score(full_df, pd.Series(all_pred_costs, index=full_df.index))

    # Plot provider scorecard
    plot_provider_scorecard(scorecard, top_n=20)

    # Save scorecards
    scorecard.to_csv(OUTPUT_DIR / "reports" / "provider_scorecard.csv", index=False)
    provider_risk.to_csv(OUTPUT_DIR / "reports" / "provider_upcoding_risk.csv", index=False)

    # ── STEP 7: Feature importance / Explainability ───────────────────────────
    print("\n[7/8] Computing feature importance...")

    # Cost model explainer
    print("  → Cost prediction importance...")
    cost_explainer = FeatureExplainer(
        cost_model.model,
        cost_data["X_train"].sample(min(200, len(cost_data["X_train"])), random_state=SEED),
        model_name="Cost Prediction"
    )
    cost_explainer.compute(cost_data["X_test"].sample(min(500, len(cost_data["X_test"])), random_state=SEED))
    cost_explainer.plot_importance_bar(top_n=15)
    print(cost_explainer.global_importance(10).to_string(index=False))

    # Readmission model explainer — use base_model (pre-calibration)
    print("\n  → Readmission importance...")
    readmit_explainer = FeatureExplainer(
        clf.base_model,
        readmit_data["X_train"].sample(min(200, len(readmit_data["X_train"])), random_state=SEED),
        model_name="Readmission Risk"
    )
    readmit_explainer.compute(
        readmit_data["X_test"].sample(min(500, len(readmit_data["X_test"])), random_state=SEED)
    )
    readmit_explainer.plot_importance_bar(top_n=15)
    print(readmit_explainer.global_importance(10).to_string(index=False))

    # Example: explain one high-risk episode
    high_risk_idx = risk_results[risk_results["risk_band"] == "Very High"].index
    if len(high_risk_idx) > 0:
        sample_X = readmit_data["X_test"].sample(min(500, len(readmit_data["X_test"])), random_state=SEED)
        common   = [i for i in high_risk_idx if i in sample_X.index]
        if common:
            pos = sample_X.index.get_loc(common[0])
            explanation = readmit_explainer.explain_episode(pos)
            print(f"\n  Episode explanation (Very High readmission risk):")
            print(explanation[["feature", "value", "shap_value", "direction"]].to_string(index=False))

    # ── STEP 8: Save models + summary report ─────────────────────────────────
    print("\n[8/8] Saving models and generating report...")
    cost_model.save()
    clf.save()
    detector.save()
    benchmarker.save()

    all_metrics = {
        "Cost Prediction (XGBoost Regressor)": cost_metrics,
        "Readmission Classifier (XGBoost + Calibration)": readmit_metrics,
        "Anomaly Detector (Isolation Forest)": anomaly_metrics,
        "Provider Benchmarking": {
            "n_providers":     len(scorecard),
            "peer_groups":     benchmarker.n_clusters,
            "cost_outliers":   int(scorecard["cost_outlier"].sum()),
            "los_outliers":    int(scorecard["los_outlier"].sum()),
            "readmit_outliers":int(scorecard["readmit_outlier"].sum()),
        }
    }
    save_summary_report(all_metrics)

    print("\n" + "=" * 60)
    print("  Pipeline complete!")
    print(f"  Models   → outputs/models/")
    print(f"  Plots    → outputs/plots/")
    print(f"  Reports  → outputs/reports/")
    print("=" * 60)

    return {
        "episodes_df":    episodes_df,
        "scorecard":      scorecard,
        "cost_results":   cost_results,
        "risk_results":   risk_results,
        "anomaly_results":anomaly_results,
        "provider_risk":  provider_risk,
        "metrics":        all_metrics,
    }

In [8]:
if __name__ == "__main__":
    results = run_pipeline()

  DRG Analytics DS Pipeline

[1/8] Generating synthetic data...
Generating 5,000 synthetic DRG episodes...


100%|██████████| 5000/5000 [00:02<00:00, 1885.81it/s]



✓ Generated 5,000 episodes across 40 providers
  Readmission rate:  18.9%
  Upcoding rate:     7.3%
  Mean episode cost: $22,972
  Median LoS:        7.0 days

[2/8] Preprocessing & feature engineering...

── Preprocessing summary ──────────────────────────────
  Total episodes:        5,000
  Cost features:         43
  Readmission features:  44
  Train: 3,500  Val: 500  Test: 1,000
  Readmission rate (train): 18.9%

[3/8] Training cost prediction model (XGBoost Regressor)...
  Using backend: xgboost
✓ Cost prediction model fitted

── Cost Predictor Evaluation ───────────────────────
  Train RMSE: 0.0664  |  Test RMSE: 0.1780
  Train MAE:  0.0495  |  Test MAE:  0.1416
  Train R²:   0.9855  |  Test R²:   0.8887
  Cost outliers flagged: 51  (5.1% of test set)

[4/8] Training readmission classifier (XGBoost + Calibration)...
  Using backend: xgboost
  ✓ Probability calibration applied (Platt scaling)
  Optimal threshold: 0.355  (val F1=1.000)
✓ Readmission classifier fitted

── Readmiss